In [ ]:
import numpy as np
import scipy as sp
import pandas as pd
import seaborn as sb
import matplotlib.pyplot as plt

In [ ]:
from google.colab import files

uploaded = files.upload()

Saving archive.zip to archive (1).zip


In [ ]:
import zipfile

with zipfile.ZipFile("archive.zip", "r") as zip_ref:
    zip_ref.extractall("olist_data")

In [ ]:
import pandas as pd

orders = pd.read_csv("olist_data/olist_orders_dataset.csv")
orders.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


In [ ]:
path = "olist_data/"

customers = pd.read_csv(path + "olist_customers_dataset.csv")
geolocation = pd.read_csv(path + "olist_geolocation_dataset.csv")
order_items = pd.read_csv(path + "olist_order_items_dataset.csv")
payments = pd.read_csv(path + "olist_order_payments_dataset.csv")
reviews = pd.read_csv(path + "olist_order_reviews_dataset.csv")
orders = pd.read_csv(path + "olist_orders_dataset.csv")
products = pd.read_csv(path + "olist_products_dataset.csv")
sellers = pd.read_csv(path + "olist_sellers_dataset.csv")
category_translation = pd.read_csv(path + "product_category_name_translation.csv")

In [ ]:
datasets = {
    "Customers": customers,
    "Geolocation": geolocation,
    "Order Items": order_items,
    "Payments": payments,
    "Reviews": reviews,
    "Orders": orders,
    "Products": products,
    "Sellers": sellers,
    "Category Translation": category_translation
}

for name, df in datasets.items():
    print(f"{name}: {df.shape[0]} rows, {df.shape[1]} columns")

Customers: 99441 rows, 5 columns
Geolocation: 1000163 rows, 5 columns
Order Items: 112650 rows, 7 columns
Payments: 103886 rows, 5 columns
Reviews: 99224 rows, 7 columns
Orders: 99441 rows, 8 columns
Products: 32951 rows, 9 columns
Sellers: 3095 rows, 4 columns
Category Translation: 71 rows, 2 columns


In [ ]:
orders.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype 
---  ------                         --------------  ----- 
 0   order_id                       99441 non-null  object
 1   customer_id                    99441 non-null  object
 2   order_status                   99441 non-null  object
 3   order_purchase_timestamp       99441 non-null  object
 4   order_approved_at              99281 non-null  object
 5   order_delivered_carrier_date   97658 non-null  object
 6   order_delivered_customer_date  96476 non-null  object
 7   order_estimated_delivery_date  99441 non-null  object
dtypes: object(8)
memory usage: 6.1+ MB


In [ ]:
for name, df in datasets.items():
    print("\n", name)
    missing = df.isnull().sum()
    print(missing[missing > 0])


 Customers
Series([], dtype: int64)

 Geolocation
Series([], dtype: int64)

 Order Items
Series([], dtype: int64)

 Payments
Series([], dtype: int64)

 Reviews
review_comment_title      87656
review_comment_message    58247
dtype: int64

 Orders
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
dtype: int64

 Products
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64

 Sellers
Series([], dtype: int64)

 Category Translation
Series([], dtype: int64)


In [ ]:
for name, df in datasets.items():
    print(f"{name}: {df.duplicated().sum()} duplicates")

Customers: 0 duplicates
Geolocation: 261831 duplicates
Order Items: 0 duplicates
Payments: 0 duplicates
Reviews: 0 duplicates
Orders: 0 duplicates
Products: 0 duplicates
Sellers: 0 duplicates
Category Translation: 0 duplicates


In [ ]:
date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for col in date_columns:
    orders[col] = pd.to_datetime(orders[col], errors="coerce")

In [ ]:
geolocation = geolocation.drop_duplicates().reset_index(drop=True)

In [ ]:
products = products.merge(
    category_translation,
    on="product_category_name",
    how="left"
)

In [ ]:
orders["delivery_days"] = (
    orders["order_delivered_customer_date"]
    - orders["order_purchase_timestamp"]
).dt.total_seconds() / (24 * 60 * 60)

orders["delivery_delay_days"] = (
    orders["order_delivered_customer_date"]
    - orders["order_estimated_delivery_date"]
).dt.total_seconds() / (24 * 60 * 60)

orders["delivery_status"] = np.where(
    orders["order_delivered_customer_date"].isna(),
    "Not Delivered",
    np.where(
        orders["delivery_delay_days"] > 0,
        "Late",
        "On Time"
    )
)

In [ ]:
orders["order_year"] = orders["order_purchase_timestamp"].dt.year
orders["order_month"] = orders["order_purchase_timestamp"].dt.to_period("M").astype(str)

In [ ]:
order_item_summary = (
    order_items.groupby("order_id")
    .agg(
        total_items=("order_item_id", "count"),
        total_product_value=("price", "sum"),
        total_freight_value=("freight_value", "sum")
    )
    .reset_index()
)

In [ ]:
payment_summary = (
    payments.groupby("order_id")
    .agg(
        total_payment_value=("payment_value", "sum"),
        payment_installments=("payment_installments", "max")
    )
    .reset_index()
)

In [ ]:
payment_summary = (
    payments.groupby("order_id")
    .agg(
        total_payment_value=("payment_value", "sum"),
        payment_installments=("payment_installments", "max")
    )
    .reset_index()
)

In [ ]:
review_summary = (
    reviews.groupby("order_id")
    .agg(
        review_score=("review_score", "mean")
    )
    .reset_index()
)

In [ ]:
processed_data = orders.copy()

processed_data = processed_data.merge(
    order_item_summary,
    on="order_id",
    how="left"
)

processed_data = processed_data.merge(
    payment_summary,
    on="order_id",
    how="left"
)

processed_data = processed_data.merge(
    review_summary,
    on="order_id",
    how="left"
)

In [ ]:
print("Rows:", processed_data.shape[0])
print("Columns:", processed_data.shape[1])

print("\nDuplicates:", processed_data.duplicated().sum())

print("\nMissing values:")
print(
    processed_data.isnull().sum()
    [processed_data.isnull().sum() > 0]
)

Rows: 99441
Columns: 19

Duplicates: 0

Missing values:
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
delivery_days                    2965
delivery_delay_days              2965
total_items                       775
total_product_value               775
total_freight_value               775
total_payment_value                 1
payment_installments                1
review_score                      768
dtype: int64


In [ ]:
processed_data.to_csv(
    "processed_olist_orders.csv",
    index=False
)

print("Processed data saved successfully!")

Processed data saved successfully!


In [ ]:
print("Total Orders:", processed_data["order_id"].nunique())
print("Total Product Value:", processed_data["total_product_value"].sum())
print("Average Review Score:", processed_data["review_score"].mean())
print("On-Time Delivery %:",
      (processed_data["delivery_status"].eq("On Time").sum() /
       processed_data["delivery_status"].isin(["On Time","Late"]).sum()) * 100)

Total Orders: 99441
Total Product Value: 13591643.700000001
Average Review Score: 4.0867934152875325
On-Time Delivery %: 91.88710145528421


In [ ]:
print(processed_data.groupby("order_month")["total_product_value"].sum().tail(10))

# Top categories
print(products.groupby("product_category_name_english")["product_id"]
      .count().sort_values(ascending=False).head(10))

# Delivery vs review
print(processed_data.groupby("delivery_status")["review_score"].mean())

# Repeat customers
# Merge orders with customers to get customer_unique_id
orders_with_customers = orders.merge(
    customers[["customer_id", "customer_unique_id"]],
    on="customer_id",
    how="left"
)
customer_orders = orders_with_customers.groupby("customer_unique_id").size()
print("Repeat customer %:",
      (customer_orders > 1).mean() * 100)

order_month
2018-01    950030.36
2018-02    844178.71
2018-03    983213.44
2018-04    996647.75
2018-05    996517.68
2018-06    865124.31
2018-07    895507.22
2018-08    854686.33
2018-09       145.00
2018-10         0.00
Name: total_product_value, dtype: float64
product_category_name_english
bed_bath_table           3029
sports_leisure           2867
furniture_decor          2657
health_beauty            2444
housewares               2335
auto                     1900
computers_accessories    1639
toys                     1411
watches_gifts            1329
telephony                1134
Name: product_id, dtype: int64
delivery_status
Late             2.566562
Not Delivered    1.753254
On Time          4.294151
Name: review_score, dtype: float64
Repeat customer %: 3.1187562437562435


In [ ]:
category_analysis = (
    order_items.merge(products[["product_id","product_category_name_english"]],
                      on="product_id", how="left")
    .groupby("product_category_name_english")
    .agg(
        revenue=("price","sum"),
        orders=("order_id","nunique")
    )
    .sort_values("revenue", ascending=False)
)

print(category_analysis.head(10))

                                  revenue  orders
product_category_name_english                    
health_beauty                  1258681.34    8836
watches_gifts                  1205005.68    5624
bed_bath_table                 1036988.68    9417
sports_leisure                  988048.97    7720
computers_accessories           911954.32    6689
furniture_decor                 729762.49    6449
cool_stuff                      635290.85    3632
housewares                      632248.66    5884
auto                            592720.11    3897
garden_tools                    485256.46    3518


In [ ]:
delivery_review = processed_data[
    processed_data["delivery_status"].isin(["On Time", "Late"])
].groupby("delivery_status")["review_score"].agg(["count","mean"])

print(delivery_review)

print("\nDifference in average rating:",
      delivery_review.loc["On Time","mean"] -
      delivery_review.loc["Late","mean"])

                 count      mean
delivery_status                 
Late              7662  2.566562
On Time          88168  4.294151

Difference in average rating: 1.7275883057642907


In [ ]:
processed_data.to_csv("processed_olist_orders.csv", index=False)

In [ ]:
from google.colab import files

files.download("processed_olist_orders.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>